# HeroSoftware: where the next 1,000 merchants come from
Independent case study based on public data · not affiliated with or endorsed by HeroSoftware GmbH or Shopify.

**PUBLIC DATA** = Shopify App Store pages, Eurostat, ECB (observed 2026-11-23). **SIMULATED** = a fictional 4-app portfolio (APP_A-APP_D); never HeroSoftware's figures.

In [1]:
import sys, json, subprocess
from pathlib import Path
import pandas as pd
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT / 'src'))
pd.set_option('display.width', 160)
def run(script):
    r = subprocess.run([sys.executable, str(ROOT / 'src' / script)], cwd=ROOT, capture_output=True, text=True)
    print(r.stdout.strip().splitlines()[-1] if r.stdout.strip() else r.stderr[-500:])
def out(name):
    return pd.read_csv(ROOT / 'data' / 'out' / f'{name}.csv')

## 1. Collection (PUBLIC DATA)
One polite pass (robots.txt, 1 request / 1.5 s, cached). Without the raw cache, the delivered CSVs in `data/public/` are used.

In [2]:
run('collect_public.py')
for f in sorted((ROOT / 'data' / 'public').glob('*.csv')):
    print(f.name, pd.read_csv(f).shape)

done: categories.csv, company_facts.json, ecb_eurusd_monthly.csv, eurostat_b2c_online_sales.csv, holidays_de.csv, listings.csv, reviews_public.csv, wayback_rating_counts.csv
categories.csv (2568, 15)
ecb_eurusd_monthly.csv (12, 2)
eurostat_b2c_online_sales.csv (8, 5)
holidays_de.csv (18, 2)
listings.csv (4, 16)
reviews_public.csv (514, 11)
wayback_rating_counts.csv (16, 4)


## 2. The real shelf (PUBLIC DATA)

In [3]:
run('simulate_growth.py')
run('run_sql.py')
out('reviews_totals').T

by app: {'APP_A': 3782, 'APP_B': 3280, 'APP_C': 168, 'APP_D': 2127} | DE share 66.1%


30 tables exported to data/out/ | tests passed 11/11


,0
reviews,514
unique_ids,514
share_5_star,0.982
share_germany,0.85
share_dach,0.934
share_tenure_in_days,0.484
share_under_30_days,0.642
may_jun_2025,308
replies_2026_11_09,293
since_2026_03,74


In [4]:
out('reviews_by_app')[['app', 'reviews', 'share_dach', 'reply_rate', 'last_review', 'reviews_since_2026_03']]

,app,reviews,share_dach,reply_rate,last_review,reviews_since_2026_03
0,AddressHero,235,0.911,1.000,2026-08-07,11
1,PaymentHero,213,0.967,0.000,2026-02-24,0
2,Post Purchase Upsell Hero,63,0.905,0.984,2026-11-11,62
3,DiscountHero,3,1.000,0.000,2026-07-01,1


In [5]:
out('category_summary')[['herosoftware_app', 'apps_listed', 'hero_position', 'hero_review_rank', 'free_plan_top30', 'free_plan_rest']]

,herosoftware_app,apps_listed,hero_position,hero_review_rank,free_plan_top30,free_plan_rest
0,Post Purchase Upsell Hero,1002,66,151,0.667,0.265
1,AddressHero,914,7,25,0.500,0.070
2,PaymentHero,402,30,7,0.600,0.129
3,DiscountHero,250,123,70,0.567,0.132


In [6]:
out('eurostat')[['geo', 'pct_enterprises_selling_online_b2c']].T

,0,1,2,3,4,5,6,7
geo,AT,ES,SE,NL,DE,FR,IT,PL
pct_enterprises_selling_online_b2c,22.47,22.34,20.41,19.07,15.65,14.25,14.24,13.95


## 3. Simulation (SIMULATED)
Seed 42, 2025-11-01 -> 2026-10-31. Assumptions in `docs/assumptions.md`.

In [7]:
for f in sorted((ROOT / 'data' / 'sim').glob('*.parquet')):
    print(f.stem, pd.read_parquet(f).shape)
out('dq_summary')[['item', 'n']]

email_events (34421, 6)
experiment_arms (40, 5)
experiments (24, 16)
fx (12, 2)
install_events (15611, 9)
listing_traffic (100096, 5)
marketing_spend (1241, 6)
merchants (9357, 4)
reviews_sim (8198, 6)
subscriptions (7364, 8)


,item,n
0,raw installed events,10717
1,webhook retry duplicates removed,1133
2,quick re-installs (<24h) not counted as new,227
3,"net installs (clean, one per merchant x app)",9357
4,installs in a different month in UTC vs Berlin,4


## 4. Channels (SIMULATED)

In [8]:
out('channel_scorecard')[['channel', 'installs', 'uninstall_48h_rate', 'trial_to_paid', 'spend_eur', 'cac_per_paying', 'payback_months']]

,channel,installs,uninstall_48h_rate,trial_to_paid,spend_eur,cac_per_paying,payback_months
0,partner,297,0.054,0.423,7750.0,75.0,4.4
1,google_ads_brand,298,0.128,0.209,7037.0,144.0,8.5
2,content,423,0.142,0.224,4950.0,67.0,3.8
3,google_ads_nonbrand,115,0.113,0.253,2955.0,118.0,7.2
4,app_store_ads,170,0.406,0.054,1747.0,349.0,30.7
5,email,126,0.127,0.293,1500.0,52.0,2.7
6,app_store_search,2774,0.113,0.229,0.0,NaN,NaN
7,category_browse,1477,0.122,0.244,0.0,NaN,NaN
8,direct,976,0.123,0.235,0.0,NaN,NaN
9,social,297,0.125,0.238,0.0,NaN,NaN


## 5. Experiments (SIMULATED)

In [9]:
ro = out('experiment_readouts')
print(ro.decision.value_counts().to_dict())
ro[['exp_id', 'title', 'lift', 'ci_low', 'ci_high', 'rule_call', 'decision']].round(2).head(8)

{'Scale': 9, 'Kill': 9, 'Rerun': 3, 'Inconclusive': 3}


,exp_id,title,lift,ci_low,ci_high,rule_call,decision
0,EXP01,Review request at first value moment,1.11,0.33,1.89,Scale,Scale
1,EXP02,LinkedIn content push in BFCM week,-0.14,-0.38,0.19,Inconclusive,Rerun
2,EXP03,Listing translated into French and Italian,0.38,-0.04,0.98,Inconclusive,Scale
3,EXP04,Shopify App Store ads,-0.85,-0.98,-0.71,Kill,Kill
4,EXP05,Google Ads brand: 4-week geo holdout,-0.11,-0.20,-0.03,Kill,Kill
5,EXP06,Meta prospecting ads (3 weeks),0.06,-0.12,0.23,Inconclusive,Inconclusive
6,EXP07,In-app cross-promotion to APP_D,1.82,0.47,3.17,Scale,Scale
7,EXP08,Win-back e-mail after uninstall,-0.05,-0.28,0.18,Inconclusive,Kill


In [10]:
out('exp03_did')

,metric,t_pre,t_post,c_pre,c_post,naive_before_after,control_change,did_lift,did_ci_low,did_ci_high
0,installs,86.0,173.0,98.0,143.0,0.496,0.085,0.379,-0.043,0.985
1,listing_views,1194.0,2450.0,1463.0,2127.0,0.526,0.081,0.411,0.282,0.554


In [11]:
out('exp04_quality')

,cohort,installs,uninstall_48h,trials,paying,trial_to_paid
0,App Store ads,170,0.406,93,5,0.054
1,Organic + other,599,0.122,462,116,0.251


## 6. Traps (SIMULATED)

In [12]:
display(out('bfcm_did')); display(out('brand_holdout')[['claimed', 'incremental_share', 'incr_ci_low', 'incr_ci_high', 'incremental_conversions']])
out('simpson').query("level != 'country'")

,naive_lift,not_promoted_change,did_lift,did_ci_low,did_ci_high
0,0.442,0.675,-0.139,-0.377,0.189


,claimed,incremental_share,incr_ci_low,incr_ci_high,incremental_conversions
0,410.0,0.114,0.025,0.196,47.0


,level,segment,app,trials,trial_to_paid
0,region,DACH,APP_B,1861,0.279
1,region,DACH,APP_D,659,0.316
2,region,non-DACH,APP_B,705,0.081
3,region,non-DACH,APP_D,751,0.104
4,overall,all,APP_B,2566,0.225
5,overall,all,APP_D,1410,0.203


In [13]:
out('tests')

,test,passed
0,no duplicate merchant x app after cleaning,True
1,no retry duplicate left in stg_events,True
2,"514 public reviews, unique ids",True
3,reviews = sum of listing ratingCount,True
4,avg stars per app within 0.05 of listing,True
5,public review table has no shop name / text co...,True
6,no published country with fewer than 5 reviews,True
7,installs >= trials >= paying (every channel),True
8,funnel views >= installs (every app x surface),True
9,24 experiments,True


## 7. Deliverables and plan
Excel tracker (verified by recalculation), dashboard, PDF. The plan is in `docs/90_day_growth_plan.md`.

In [14]:
for s in ('build_excel.py', 'build_dashboard.py', 'build_pdf.py'):
    run(s)

verification: PASS


dashboard/index.html: 5.0 MB, 5 tabs


PDF: 7 pages -> report/HeroSoftware_growth_case_study.pdf
